# 🪟 **Reading by Windows & Decimation**

> ### 📌 **TL;DR**
>
> This Jupyter notebook has been created to compare different features with several open source Python libraries for rasters management.
>
> This notebook explores windowed reading and raster decimation mechanisms across several libraries.
>
> The following libraries will be considered :
>- `rasterio`
>- `rioxarray`
>- `odc-geo`
>- `geoutils`

In [ ]:
import rasterio
from rasterio.windows import Window
from rasterio.plot import show
from rasterio.enums import Resampling

import rioxarray

import matplotlib.pyplot as plt
import matplotlib.patches as patches

import seaborn #cmap mako

In [ ]:
#path to the raster object
raster_path = "../data/rasters/105005005BDEA700-visual.tif"

## **Reading by Windows**

A Window is a subset of a raster.

It is particularly useful when working with big files to reduce memory usage and speed up processing times.

### • *rasterio*

In [ ]:
ds_rasterio = rasterio.open(raster_path)

In rasterio, we can use the object [`Window`](https://rasterio.readthedocs.io/en/stable/api/rasterio.windows.html) to create one.

In the following case, we will create a subset of our image with `1500 rows x 1500 columns`.

In [ ]:
show(ds_rasterio.read(1), cmap="mako")

In [ ]:
window = Window(col_off=1500, row_off=1500, width=1500, height=1500) #1500 x 1500

We will focus on the first band for our example.

In [ ]:
b1 = ds_rasterio.read(1)

w = ds_rasterio.read(1, window=window)

Let's compare the full raster with the subset image created with the Window:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

#full raster
axes[0].imshow(b1, cmap="mako")
axes[0].set_title("Full Raster")
axes[0].axis("off")

#rectangle
rect = patches.Rectangle(
    (window.col_off, window.row_off),
    window.width,
    window.height,
    linewidth=2, edgecolor='red', facecolor='none'
)
axes[0].add_patch(rect)

#window raster
axes[1].imshow(w, cmap="mako")
axes[1].set_title("Window (subset)")
axes[1].axis("off")

plt.show()

### • *rioxarray*

In [ ]:
ds_rioxarray = rioxarray.open_rasterio(raster_path)

With rioxarray, we can use [`.isel_window()`](https://corteva.github.io/rioxarray/stable/rioxarray.html#rioxarray.rioxarray.XRasterBase.isel_window) to create a subset of our image. 

As an argument, we will pass our `Window` previously created to select the extents of the new raster :

In [ ]:
window

In [ ]:
rioxarray_subset = ds_rioxarray.rio.isel_window(window)

In [ ]:
show(rioxarray_subset.sel(band=1), cmap="mako")

- ### *geoutils*

In geoutils, this feature has an [opened issue](https://github.com/GlacioHack/geoutils/issues/583).

Creating a window through geoutils uses a different mechanism from other libraries. The right way for that is to open the data object and then to use [`.crop()`](https://geoutils.readthedocs.io/en/stable/gen_modules/geoutils.Raster.crop.html#geoutils.Raster.crop) or [`.icrop()`](https://geoutils.readthedocs.io/en/stable/gen_modules/geoutils.Raster.icrop.html#geoutils.Raster.icrop).

This method is more detailed in the [`mask_and_crop`](https://github.com/sertit/open_source_rasters_comparison/blob/main/notebooks/04_mask_and_crop.ipynb) notebook.

## **Decimation**

Decimation is used to reduce the image resolution of a raster by skipping a certain amount of pixels in rows, columns or both.

It is particularly useful to speed up the display or analyze large raster files.

### • *rasterio*

In the following case, we will reduce the size by half (height and width). 

Our raster's size is `17408 x 17408`, so we want to reshape it as `8704 x 8704`. 

We can do so by using the [`.read()`](https://rasterio.readthedocs.io/en/stable/api/rasterio._io.html#rasterio._io.DatasetReaderBase.read) function, but we will use the *out_shape* argument to select the extent we want to keep :

In [ ]:
decimated_b1 = ds_rasterio.read(1, out_shape=(8704, 8704))

We can now plot the decimated raster :

In [ ]:
show(decimated_b1, cmap="mako")

### • *rioxarray*

There is no trivial method to decimate an image through xarray and especially rioxarray.

To do so, we first have to `reproject` it and then use a `resampling` algorithm, as shown below:

In [ ]:
da = rioxarray.open_rasterio(raster_path)

Here we specify our new wanted resolution. We will multiply it by 2, so the final size of our raster will be divided by 2:

In [ ]:
new_res = (
    da.rio.resolution()[0] * 2,
    da.rio.resolution()[1] * 2
)

Finally, we will use [`.reproject()`](https://corteva.github.io/rioxarray/stable/rioxarray.html#rioxarray.raster_array.RasterArray.reproject) and we specify the original CRS, the new resolution and the resampling method (`average` here):

In [ ]:
da_decimated = da.rio.reproject(
    da.rio.crs,
    resolution=new_res,
    resampling=Resampling.average
)

We can finally plot the new map:

In [ ]:
da_decimated.sel(band=1).plot(cmap="mako")